# KURE-v1 건강 상담 문서 임베딩 (Colab GPU)

`scripts/compare_kure.py`가 쓸 KURE-v1(`nlpai-lab/KURE-v1`, BGE-M3를 한국어 검색으로 추가 학습, 1024차원) 문서 벡터를 만든다.

- 길이 제한 1,024토큰: 상담 글 19,206건 중 0.11%만 잘린다(512토큰이면 2.4%).
- 접두어 없이 넣는다(모델 카드 규약).
- 런타임: GPU(T4 이상). 모델 약 2.3GB를 받는다.
- HF 토큰: Colab Secrets의 `HF_TOKEN` (없으면 실행 중 입력)
- 결과: 비공개 데이터셋 `blanden77/dograg-data`의 `experiments` 브랜치에 `kure/kure_v1_rows.npy`로 올린다(벡터만). 행 순서 = `df.csv` 행 번호 = 앱의 문서 ID.

In [ ]:
!pip -q install -U sentence-transformers huggingface_hub
import torch; print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'GPU 없음: 런타임 유형을 GPU로 바꾸세요')

In [ ]:
import getpass
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
except Exception:
    HF_TOKEN = None
HF_TOKEN = (HF_TOKEN or getpass.getpass('HF_TOKEN (영문 입력 상태에서 붙여 넣기): ')).strip()
# A token typed with the Korean input mode on turns into Hangul and fails later as UnicodeEncodeError.
if not HF_TOKEN.isascii() or not HF_TOKEN.startswith('hf_'):
    raise ValueError(f'HF_TOKEN이 hf_로 시작하는 영문·숫자 값이 아닙니다(길이 {len(HF_TOKEN)}, 한글 포함: {not HF_TOKEN.isascii()}). '
                     'Hugging Face 토큰을 복사해 Colab Secrets의 HF_TOKEN에 붙여 넣으세요.')
print('HF_TOKEN ok, length', len(HF_TOKEN))

In [ ]:
import pandas as pd
from huggingface_hub import hf_hub_download
REPO, REVISION = 'blanden77/dograg-data', 'data-v3'  # src/private_data.py
path = hf_hub_download(REPO, 'data/df.csv', repo_type='dataset', revision=REVISION, token=HF_TOKEN)
df = pd.read_csv(path)
# The app's documents are qa.input with carriage returns removed (19,206 of 19,206 match the Chroma texts).
CR = chr(13)
texts = df['qa.input'].astype(str).str.replace(CR, '', regex=False).tolist()
print(len(texts))

In [ ]:
import time, numpy as np
from sentence_transformers import SentenceTransformer
model = SentenceTransformer('nlpai-lab/KURE-v1', device='cuda').half()  # float16 on the GPU: about twice as fast
model.max_seq_length = 1024
# Shortest first: batches of similar length waste less padding.
order = np.argsort([len(t) for t in texts])
started = time.time()
sorted_vectors = model.encode([texts[i] for i in order], batch_size=64, normalize_embeddings=True,
                              show_progress_bar=True, convert_to_numpy=True).astype(np.float32)
vectors = np.empty_like(sorted_vectors)
vectors[order] = sorted_vectors
print(vectors.shape, f'{time.time() - started:.0f}s', 'norm', float(np.linalg.norm(vectors[:3], axis=1).mean()))
np.save('kure_v1_rows.npy', vectors)

In [ ]:
from huggingface_hub import HfApi
api = HfApi(token=HF_TOKEN)
api.create_branch(REPO, repo_type='dataset', branch='experiments', exist_ok=True)
api.upload_file(path_or_fileobj='kure_v1_rows.npy', path_in_repo='kure/kure_v1_rows.npy',
                repo_id=REPO, repo_type='dataset', revision='experiments')
print('uploaded to', REPO, '@ experiments')